In [ ]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
"""Spectrum 1D ON/OFF Analysis"""

# Spectrum 1D ON/OFF Analysis with 1LHAASO Catalog


This notebook demonstrates how to:
1. Import necessary libraries and modules.
2. Load and select a source from the 1LHAASO catalog.
3. Configure an `AnalysisSpectrumConfig` object.
4. Perform spectrum analysis using `AnalysisSpectrum`.
5. Save and inspect the results, including flux points and fit results.

---


In [1]:
# Standard library imports
import astropy.units as u

# Third-party imports from gammapy
from gammapy.catalog import SourceCatalog1LHAASO
from gammapy.data import Observation
from gammapy.datasets import Datasets
from gammapy.utils.scripts import make_path
from gammapy.modeling.models import PowerLawSpectralModel, SkyModel, Models

# Third-party imports from regions
from regions import CircleSkyRegion

# feupy module imports
from feupy.utils.coordinates import convert_skycoord_to_dict
from feupy.analysis.config import CTAOAnalysisConfig
from feupy.analysis.core import CTAOAnalysis
from feupy.visualization.counts import show_hist_counts

from feupy.irf import (
CTAOVisibilityEstimator,
make_ctao_visibility_table,
)

### 1. Load and Select Source
In this section, we load the 1LHAASO catalog and select a specific source (1LHAASO J1219+2915) for analysis.

In [ ]:
# Load the 1LHAASO catalog and select a source
catalog = SourceCatalog1LHAASO()
catalog.table

In [ ]:
source = catalog["1LHAASO J2002+3244u"]

In [ ]:
source.position

In [ ]:
source.name

In [ ]:
source.sky_model()

In [ ]:
source.spectral_model("KM2A")

In [ ]:
source.spectral_model("WCDA")

### Annual Visibility

In [ ]:
year=2026

estimator = CTAOVisibilityEstimator(
    target=source.position,
    year=year,
    time_step_min=30,
)

df = make_ctao_visibility_table(estimator)

df

### 2. Configure `CTAOAnalysis`
Here, we configure the `CTAOAnalysisConfig` object with observation, dataset, and analysis settings for the spectrum analysis.

In [ ]:
config = CTAOAnalysisConfig()

position = source.position.icrs

config.observation.obs_cone = convert_skycoord_to_dict(position)
config.observation.position_angle = 0 * u.deg
config.observation.offset = 0.5 * u.deg
config.observation.livetime = 50 * u.h
config.observation.required_irfs = ["South", "AverageAz", "40deg", "50h"]

config.datasets.map_selection = ["edisp", "background", "exposure"]

config.datasets.safe_mask.methods = ["aeff-default"]
config.datasets.safe_mask.parameters = {"aeff_percent": 10}

config.datasets.containment_correction = False
config.datasets.use_region_center = True

config.datasets.on_region = convert_skycoord_to_dict(position)
config.datasets.on_region.radius = 0.2 * u.deg

config.datasets.stack = False

config.datasets.on_off.acceptance = 1
config.datasets.on_off.acceptance_off = 5

config.datasets.geom.axes.energy.min = 3 * u.GeV
config.datasets.geom.axes.energy.max = 100 * u.TeV
config.datasets.geom.axes.energy.nbins = 15

config.datasets.geom.axes.energy_true.min = .3 * u.GeV
config.datasets.geom.axes.energy_true.max = 300 * u.TeV
config.datasets.geom.axes.energy_true.nbins = 20

config.flux_points.energy.min = 3 * u.GeV
config.flux_points.energy.max = 100 * u.TeV
config.flux_points.energy.nbins = 15
config.flux_points.source = "source"

config.sensitivity.gamma_min = 3
config.sensitivity.n_sigma = 2
config.sensitivity.bkg_syst_fraction = 0.05

config.statistics.n_obs = 1

DATA_PATH = make_path("./data/")
DATA_PATH.mkdir(parents=True, exist_ok=True)

config.sensitivity.data_path = DATA_PATH

print(config)

### 3. Running Spectrum Analysis
In this section, we create an instance of `CTAOAnalysis` and perform the spectrum analysis based on the configured settings. The process includes simulating observations, running fits, and extracting flux points.

In [ ]:
# Create and run spectrum analysis
analysis = CTAOAnalysis(config)

# Simulate observation and spectrum
analysis.simulate_observation()

In [ ]:
source.spectral_model("WCDA")

In [ ]:
model_simu =  PowerLawSpectralModel(
    index=2.21,
    amplitude=3.4e-14 * u.Unit("cm-2 s-1 TeV-1"),
    reference=3* u.TeV,
)
model_source = SkyModel(spectral_model=model_simu, name="source")
analysis.get_spectrum_dataset(model_source)

In [ ]:
print(analysis.spectrum_dataset)

In [ ]:
analysis.get_datasets()

In [ ]:
print(analysis.datasets)

In [ ]:
analysis.datasets.info_table()

In [ ]:
analysis.set_models(Models(model_source))

In [ ]:
print(analysis.datasets)

In [ ]:
analysis.get_file_name()

In [ ]:
analysis.run_fit()

In [ ]:
analysis.fit_result.parameters.to_table()

In [ ]:
analysis.get_flux_points()

In [ ]:
analysis.flux_points

In [ ]:
# Run the sensitivity analysis
analysis.compute_sensitivity()

# Write the sensitivity table to a file (CSV in this case)
analysis.write_table_sensitivity()

# Read the sensitivity table back if needed
sensitivity_table = analysis.read_table_sensitivity()

# Print the sensitivity table summary
print("\nSensitivity Table:")
print(sensitivity_table)

In [ ]:
table_sens = analysis.table_sens

In [ ]:
from feupy.visualization.sensitivity import plot_sensitivity_from_table

plot_sensitivity_from_table(table_sens)

In [ ]:
table = analysis.table_sens
print(table[['e_ref', 'background', 'excess']])

In [ ]:
table = analysis.datasets.info_table()
table

In [ ]:
show_hist_counts(table)

In [ ]:
analysis.flux_points.data.to_table()

### 4. Inspect Results
Here, we review the results of the spectrum analysis, including the fit parameters and computed flux points.

In [ ]:
# Display the fit parameters
print("Fit Parameters:")
print(analysis.fit_result)

# Display the flux points
print("Flux Points:")
print(analysis.flux_points)

In [ ]:
analysis.flux_points.plot_fit()

In [ ]:
from astropy.table import  Table

In [ ]:
table = Table.read("sens_CTAO-South_60deg_50h_livetime50.0h.fits")

In [ ]:
model = analysis.fit_result.models[0]

In [ ]:
from feupy.visualization.sensitivity import plot_irfs

In [ ]:
model

In [ ]:
plot_irfs([table_sens])

In [ ]:
ax = plot_irfs([table_sens], model, energy_bounds=[30 * u.GeV, (100 * u.TeV).to('GeV')])
fig = ax.get_figure()
fig.savefig("sens_CTAO.png", dpi=300, bbox_inches="tight")

## 5. Summary

In this notebook, we:

1. Selected a source from the 3HWC catalog.
2. Configured and ran a 1D ON/OFF spectrum analysis using `AnalysisSpectrum`.
3. Printed and saved the fit results and flux points for further analysis.

This modular design
